#Ingest the Drivers csv file
1. Read the file using spark DataframeReader API
2. Add metadata columns 
   - Source File
   - Ingestion Timestamp
3. Write to Bronze Delta Tables


### Read the CSV using Spark Dataframe API


In [0]:
%run ../00_common/01_environment_config


In [0]:
%run ../00_common/02_bronze_helpers

In [0]:
table_name = f"{catalog_name}.{bronze_schema}.drivers"
source_file = f"{landing_folder_path}/drivers.json"

In [0]:
from pyspark.sql.types import StructType,StructField,StringType,DateType

name_schema = StructType([
    StructField("givenName",StringType()),
    StructField("familyName",StringType())
])

drivers_schema = StructType([
    StructField("driverId",StringType()),
    StructField("name",name_schema),
    StructField("dateOfBirth", DateType()),
    StructField("nationality",StringType()),
    StructField("url",StringType())
])


In [0]:
drivers_df=(
    spark.read
    .format("json")
    #.option("header","True")
    #.option("inferSchema", "True")
    .option("mode", "FAILFAST")
    .schema(drivers_schema)
    .load(source_file))


In [0]:
display(drivers_df)

### Adding 2 new colums
- Source File
- Ingestion Timestamp


In [0]:

drivers_final_df = new_columns_insertion(drivers_df)
display(drivers_final_df)

In [0]:
(drivers_final_df
.write
.format("delta")
.mode("overwrite")
.saveAsTable(table_name))

In [0]:
display(spark.table(table_name))